In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

In [ ]:
from turkish_tokenizer import core as tt

tokens, ids = tt.tokenize("Merhaba dünya")
print(tokens)  # ['merhaba', '<space>', 'dünya']
print(ids)     # [2036, 1, 224]

In [ ]:
vocab_dict = {**tt.bpe_tokens, **tt.suffixes, **tt.roots}
len(vocab_dict)

In [ ]:
from llama3_2.config import LlamaConfig
from llama3_2.model_trfmrs import LlamaModel

teacher_config = LlamaConfig()

teacher_model = LlamaModel(teacher_config)
teacher_model = teacher_model.to(device)
teacher_model

In [ ]:
import safetensors.torch as st

teacher_model_states = st.load_file("teacher_model1.safetensors")
teacher_model.load_state_dict(teacher_model_states)
teacher_model.eval()


In [8]:
def tr_capitalize(word):
  # i -> İ
  if word[0] == "i":
    return "İ" + word[1:]
  else:
    return word.capitalize()

In [48]:
def get_vector_for_token(token, model, tokenizer):
  token_id = 0
  if token == "<uppercase>":
    token_id = 128002
  elif token == "<space>":
    token_id = 128003
  elif token == "<newline>":
    token_id = 128011
  elif token == "<tab>":
    token_id = 128012
  elif token == "<unknown>":
    token_id = 128013
  elif token.startswith("kok_") or token.startswith("ek_") or token.startswith("special_"):
    token_id = 128014
  
  if token_id > 0:
    return model(torch.tensor([[token_id]]).to(device)).squeeze(0).squeeze(0), [token_id]

  token_ids = tokenizer.encode(token)[1:]
  token_ids0 = tokenizer.encode(tr_capitalize(token))[1:]

  if (len(token_ids)) > (len(token_ids0)):
    token_ids0 = torch.tensor([token_ids0]).to(device)
    token_vectors = model(token_ids0)
    last_token_vector = token_vectors[:, -1, :]
    return last_token_vector.squeeze(0).squeeze(0), token_ids0.squeeze(0).tolist()
  else:
    token_ids = torch.tensor([token_ids]).to(device)
    token_vectors = model(token_ids)
    last_token_vector = token_vectors[:, -1, :]
    return last_token_vector.squeeze(0).squeeze(0), token_ids.squeeze(0).tolist()


In [ ]:
from transformers import AutoTokenizer

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)

In [ ]:
new_token_tensor = llama_tokenizer.encode("merhabalar nasılsınız?")[1:]
new_token_tensor = torch.tensor([new_token_tensor]).to(device)
new_token_tensor

In [ ]:
teacher_model.embed_tokens(new_token_tensor).shape

In [ ]:
teacher_model_output = teacher_model(new_token_tensor)
teacher_model_output.shape

In [ ]:
last_token_vector = teacher_model_output[:, :-1, :]
last_token_vector.squeeze(0).shape

In [ ]:
v, token_ids = get_vector_for_token("aasdfasdf", teacher_model, llama_tokenizer)
v.shape, token_ids

In [ ]:
vocab_dict["a"]

In [ ]:
teacher_model_output[0, -1, :]

In [ ]:
from tqdm import tqdm

token_to_embeddings = []

for i in tqdm(range(len(vocab_dict.keys())), desc="Mapping embeddings to Llama embeddings"):
  token = list(vocab_dict.keys())[i]
  token_id = vocab_dict[token]
  try:
    v, token_ids = get_vector_for_token(token, teacher_model, llama_tokenizer)
    tte_row = {
      "token": token,
      "token_id": token_id,
      "llama_token_ids": token_ids,
      "embedding": v.tolist(),
    }
    token_to_embeddings.append(tte_row)
  except Exception as e:
    print(e)
    print(token, token_id)
    break


In [ ]:
import pandas as pd


# token_to_embeddings is [{"token": "a", "token_id": 128000, "llama_token_ids": [128000],  "embedding": [0.1, 0.2, 0.3, ...]}]

token_to_embeddings_df = pd.DataFrame(token_to_embeddings)
token_to_embeddings_df


In [61]:
# token_to_embeddings_df["embedding"] = token_to_embeddings_df["embedding"].apply(lambda x: x[0] if len(x) == 1 else x)

In [53]:
token_to_embeddings_df.to_csv("token_to_embeddings.csv", index=False, encoding="utf-8")

In [54]:
token_to_embeddings_df.to_pickle("token_to_embeddings.pkl")

In [ ]:
df = pd.read_pickle("token_to_embeddings.pkl")
df